# BuildingsBench-faithful load forecasting study — canonical models, one notebook

Trains + evaluates **21 models** (3 official BuildingsBench persistence baselines; LSTM, GRU;
XGBoost, LightGBM; 7 canonical published Transformer/linear architectures — PatchTST,
iTransformer, TimeXer, DLinear, Informer, Autoformer, Crossformer; the paper's **own**
pretrained model family, Transformer-S/M/L (Gaussian); and 4 novel architectures grounded in
post-BuildingsBench literature — TFTLite, xLSTM, SpectraMix, Mamba, see below) across
**2 conditions** (no-weather / +weather), on a compute-bounded subsample of the official
Buildings-900K pretraining corpus, evaluated on **both** the official simulated test set and 7
real-building datasets — plus **weather-value analyses** (extreme days, longer horizons,
sensitivity gating) and **dataset EDA figures**, all persisted to Drive.

All logic lives in the `bblab/` package (cloned from this repo below) — this notebook is a thin,
readable run log. Preprocessing (Box-Cox load transform, per-channel weather StandardScaler,
linearly-scaled calendar features, the confirmed 7-channel weather order) is sourced directly
from the official [BuildingsBench repo](https://github.com/NatLabRockies/BuildingsBench) and the
public `oedi-data-lake` S3 bucket it publishes to — not a from-scratch reimplementation.

### Metrics (paper-comparable)
Headline NRMSE/NMAE/CRPS use the paper's **published aggregation**: the **median across
per-building values** (their `evaluation/aggregate.py` `return_aggregate_median`, used by
`zero_shot.py` for every benchmark table). Per-building NRMSE/NMAE are CV(RMSE)-style
(normalized by each building's own mean load); CRPS uses the paper's own **inverse-Box-Cox
Gaussian approximation** (in kWh). A pooled/global view (`*_pooled` — the formula their
`pretrain.py` uses to monitor validation; tail-sensitive, reads much higher) is kept as a
secondary column, not for paper comparison. Every trained model's **compute budget** (params,
epochs, train/eval wall time, GPU, peak memory) is recorded per row in `sim_results_v3.csv` on
Drive, next to the weights in `results/weights/`.

### What changed vs. the previous draft of this study
- Dropped 5 homegrown architectures in favor of the field's standard published baselines, and
  added the paper's own Transformer-S/M/L (Gaussian) architecture as a direct comparison point
  (faithful port of `buildings_bench/models/transformers.py`: encoder-decoder `nn.Transformer`,
  teacher forcing at train time, autoregressive greedy decoding at inference — same shared
  train/eval loop as every other model here, no separate code path to maintain).
- Fixed: unnormalized calendar features in real-building eval, a crashed results cell
  (`PERBUILDING_REAL_DIR` NameError), single-fixed-window validation, a silently zero-filled
  corrupt training building, missing multiple-comparison correction on significance tests, and
  a mixed-units CRPS (kWh mean with Box-Cox sigma).
- Validation now matches the paper's actual split (temporal holdout of the last 2 weeks of the
  year, amy2018-release buildings only) instead of an arbitrary building-level carve-out from
  the training subsample — see the Data section below.
- Calendar encoding corrected to match the paper exactly (linear-scaled day-of-year/day-of-week/
  hour-of-day, **not** sin/cos).
- Persistence baseline is now the paper's actual 3 variants (Average/CopyLastDay/CopyLastWeek),
  not a single custom "ensemble."
- Real-building evaluation runs a headline **no-weather** track (every real building) plus an
  optional **+weather** track (temperature+humidity, wherever the public bucket has it — no
  external fetching, so buildings without published weather simply aren't in that track).

### Novel architectures (`config.NOVEL_MODELS`)
Four new models grounded in post-2023 forecasting literature, aiming to beat the canonical
baselines above rather than just replicate them. All share the same `(yh, exo) -> (mu_n,
raw_scale, mean, std)` interface as every other model, so they drop straight into the existing
train/eval loop with no special-casing.
- **`tftlite`** — a compact Temporal Fusion Transformer (Lim et al. 2021): per-timestep Variable
  Selection Networks (Gated Residual Network-weighted softmax over each input channel) feeding
  an LSTM encoder-decoder, a static enrichment GRN, then causal self-attention over the full
  history+horizon. Tests whether learned per-feature gating beats treating all exogenous inputs
  uniformly.
- **`xlstm`** — Beck et al. 2024's xLSTM sLSTM cell (exponential input/forget gates with a
  log-space stabilizer state, replacing the sigmoid gates of a vanilla LSTM) applied over 24h
  patches (7 patches per 168h window, not raw hourly steps — keeps the sequential recurrence
  affordable). Motivated by a 2024 building-heat-load benchmark where xLSTM outperformed both a
  Transformer and TFT.
- **`spectramix`** — TSMixer-style time-mixing + feature-mixing MLP blocks (Chen et al. 2023)
  over daily patches, blended with a FITS-style (Xu et al. 2024) complex-linear frequency-domain
  extrapolation branch (learned weights map the historical rFFT spectrum directly onto the
  extended-horizon spectrum), plus a differentiable heating/cooling degree-day term (learned
  balance points, ReLU degree-hours -> load correction) when weather is available. Tests whether
  a lightweight mixing+frequency approach with an explicit physical prior competes with
  attention-heavy models at a fraction of the parameter count.
- **`mamba`** — a from-scratch selective state-space scan (Gu & Dao 2023's S6: input-dependent
  discretization step size gating a diagonal linear recurrence) over daily patches, gated MLP
  around it in the Mamba block style. Tests the current "attention vs. selective-SSM" debate on
  this specific forecasting task.

### A note on compute
Transformer-L matches the paper's largest config (12+12 encoder/decoder layers, d_model=768,
~160M+ params) and is meaningfully heavier per epoch than everything else here. If you want to
skip it, run `config.ALL_MODELS.remove("transformer_l")` right after cell 2 (before cell 6) —
resumable checkpointing means it's otherwise fine to just let it span multiple Colab sessions.

### Resumability
Every long-running cell below writes to Google Drive and skips already-completed
(model, condition) pairs on re-run — safe to re-run this notebook after a Colab disconnect.
Trained weights are never re-trained once their checkpoint exists in `results/weights/`.


In [ ]:
# === 1) Mount Drive, clone repo, install deps, GPU check ===
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = "https://github.com/nm-quan/Building-Bench-Future-Works"
REPO_DIR = "/content/Building-Bench-Future-Works"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > A100."
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# === 2) Imports + config ===
from bblab import config, data, models, train, metrics, weather_real
import torch, numpy as np, pandas as pd, pickle, os, csv, time

paths = config.Paths().makedirs()
print(f"BENCH={paths.BENCH}")
print(f"models ({len(config.ALL_MODELS)}): {config.ALL_MODELS}")
print(f"L={config.L} H={config.H}  EPOCHS={config.EPOCHS} PATIENCE={config.PATIENCE}  N_TRAIN_BUILDINGS={config.N_TRAIN_BUILDINGS}")


## Data

Sources the official BuildingsBench transforms + a compute-bounded (~20k building) subsample of
Buildings-900K, the official held-out Buildings-900K-test simulated test set, and the 7
real-building datasets — all directly from the public `oedi-data-lake` S3 bucket. Cached to Drive
under `paths.BENCH` so re-running this notebook doesn't re-download/re-process from scratch.

**Validation** matches the official split exactly (`scripts/data_generation/create_index_files.py`):
a temporal holdout of the last 336h (2 weeks) of the year, on the amy2018-release buildings only
(a real chronological year -- unlike the tmy3 releases, which splice together a synthetic "typical"
year with no genuine chronological tail, and which the paper's own script never validates on).
`train.train()` samples training windows so they never cross into that holdout range for amy2018
buildings, and validates only within it -- not a same-corpus building-level carve-out.


In [ ]:
# === 3) Official BuildingsBench transforms (Box-Cox load, per-channel weather StandardScaler) ===
transforms = data.download_official_transforms(paths)
load_transform = transforms["load"]
weather_transforms = transforms["weather"]
print("weather channels (confirmed order):", list(weather_transforms.keys()))


In [ ]:
# === 4) Build the training subsample (official weekly-sampled index file) + official simulated test set ===
torch.manual_seed(config.SEED); np.random.seed(config.SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"

print("building train cache (first run downloads from S3; cached on Drive afterward)...")
TR_raw = data.build_train_cache(paths, transforms, n_buildings=config.N_TRAIN_BUILDINGS, seed=config.SEED)
print(f"  TRAIN  N={TR_raw['N']}  T={TR_raw['T']}h  n_time={TR_raw['Ft']} n_weather={TR_raw['Fw']}")

print("building simulated test cache (official Buildings-900K-test, held out from training)...")
TE_SIM_raw = data.build_sim_test_cache(paths, transforms)
print(f"  SIM TEST  N={TE_SIM_raw['N']}  T={TE_SIM_raw['T']}h")

TR = train.to_device_cache(TR_raw, DEV)
TE_SIM = train.to_device_cache(TE_SIM_raw, DEV)


In [ ]:
# === 5) Load the 7 real-building datasets directly from the public S3 bucket (no separate data-prep notebook needed) ===
TE_REAL = data.load_real_buildings(paths)
n_com = sum(1 for b in TE_REAL if b["building_type"] > 0)
n_res = sum(1 for b in TE_REAL if b["building_type"] < 0)
print(f"REAL TEST  N={len(TE_REAL)}  com={n_com}  res={n_res}")


## Dataset EDA

Publication figures for the data itself (saved to `results/figures/` on Drive): mean-normalized
load heatmaps (day-of-year x hour-of-day, Com vs Res), median daily profiles with IQR bands, a
weather-load correlation heatmap ("is there weather signal in this data at all" — worth showing
next to the weather-ablation results), and a per-dataset overview of the real-building eval set.


In [ ]:
# === 5b) Dataset EDA: load heatmaps, daily profiles, weather-load correlation, real overview ===
# All figures are saved to paths.FIGURES_DIR on Drive (200dpi PNG, manuscript-ready).
from bblab import eda

print(eda.load_heatmaps(TR_raw, paths))
print(eda.daily_profiles(TR_raw, paths))
print(eda.weather_load_correlation(TR_raw, paths))
overview, fig_path = eda.real_dataset_overview(TE_REAL, paths)
print(fig_path)
overview


## Train all models x both conditions (resumable)

`ALL_MODELS` = 3 persistence baselines (closed-form, 0 params) + LSTM/GRU/DLinear/PatchTST/
iTransformer/TimeXer/Informer/Autoformer/Crossformer (trained) + the paper's own
Transformer-S/M/L (Gaussian) (trained, teacher forcing) + XGBoost/LightGBM (fit) +
TFTLite/xLSTM/SpectraMix/Mamba (trained, `config.NOVEL_MODELS`). Skips any (model, condition)
pair already present in `sim_results_v3.csv` — safe to re-run after a disconnect. **Never
retrains a model whose checkpoint already exists on Drive** (`results/weights/{model}_{cond}.pt`
/ `.pkl`): those pairs only re-run the ~15s evaluation, so metric-definition changes never
repeat any training. Each row also records the compute budget (epochs, train/eval wall time,
GPU, peak memory).

**Capacity floor**: every transformer-family baseline (PatchTST, iTransformer, TimeXer,
Informer, Autoformer, Crossformer, TFTLite) is sized to **≥7M parameters** in both conditions
(`models.MIN_TRANSFORMER_PARAMS`), so capacity is never the reason an architecture loses to
persistence. The paper's own Transformer-S stays at its published (smaller) config on purpose —
its value is exactness, and M/L already exceed the floor. Checkpoints saved under older
(smaller) configs are detected by shape mismatch and retrained automatically; their stale rows
are dropped from the results CSV. This is the long-running cell; expect several hours (more
with Transformer-L included) for a from-scratch sweep on an A100.


In [ ]:
# === 6) Train all models x both conditions (resumable) ===
train.run_training_sweep(config.ALL_MODELS, config.CONDITIONS, TR, TE_SIM, DEV, load_transform, paths,
                          reset=False, log=True)


## Simulated-test results

Two views of the same evaluation pass:
- **Headline (paper-comparable)**: the **median across per-building values** — exactly how the
  published BuildingsBench tables aggregate (`evaluation/aggregate.py`'s
  `return_aggregate_median`, called by `zero_shot.py` for both the simulated and real-building
  benchmarks). Per-building NRMSE/NMAE are CV(RMSE)-style (normalized by that building's own
  mean load); CRPS uses the paper's inverse-Box-Cox Gaussian approximation, in kWh. These are
  the `com_nrmse`/`res_nrmse`/... columns in `sim_results_v3.csv`.
- **Pooled** (`*_pooled` columns): one pooled error over all windows of all buildings per group —
  the formula the paper's `pretrain.py` uses to monitor validation during pretraining.
  Tail-sensitive (dominated by the worst buildings), so it reads much higher than the median.
  Secondary/diagnostic view only — do **not** compare it to the paper's tables.

The bootstrap-CI table below and the Wilcoxon (Holm-Bonferroni) / Friedman + Nemenyi tests all
operate on the matched per-building arrays, mirroring the paper's bootstrapped-CI methodology.


In [ ]:
# === 7) Simulated-test results: median [95% bootstrap CI], Holm-Bonferroni-corrected significance vs persistence_avg ===
def load_pb(path):
    try:
        return pickle.load(open(path, "rb"))
    except FileNotFoundError:
        return None

BASELINE = config.BASELINE_MODEL
for cond in ["A", "B"]:
    print(f"-- Condition {cond} ({'no weather' if cond == 'A' else '+weather'}) --")
    base_pb = load_pb(f"{paths.PERBUILDING_SIM_DIR}/{BASELINE}_{cond}.pkl")
    pvals = {}
    for name in config.ALL_MODELS:
        pb = load_pb(f"{paths.PERBUILDING_SIM_DIR}/{name}_{cond}.pkl")
        if pb is None:
            continue
        vals = pb["_nrmse"][~pb["_is_res"]]
        med, lo, hi = metrics.bootstrap_median_ci(vals)
        note = ""
        if base_pb is not None and name != BASELINE:
            bvals = base_pb["_nrmse"][~base_pb["_is_res"]]
            p, better = metrics.paired_significance(vals, bvals)
            pvals[name] = p
            if better is True:
                note = f"  beats {BASELINE} (p={p:.1e}, uncorrected)"
            elif better is False and p == p and p < 0.05:
                note = f"  WORSE than {BASELINE} (p={p:.1e}, uncorrected)"
        print(f"  {name:24s} Com {med:6.2f} [{lo:5.2f},{hi:5.2f}]{note}")
    corrected = metrics.holm_bonferroni(pvals)
    winners = [k for k, (p, sig) in corrected.items() if sig]
    print(f"  Holm-Bonferroni-corrected significant wins vs {BASELINE}: {winners}\n")


In [ ]:
# === 7b) Friedman + Nemenyi across the full model set (which models differ, with multiple-comparison control) ===
for cond in ["A", "B"]:
    pbs = {name: load_pb(f"{paths.PERBUILDING_SIM_DIR}/{name}_{cond}.pkl") for name in config.ALL_MODELS}
    pbs = {k: v for k, v in pbs.items() if v is not None}
    names = list(pbs.keys())
    arrs = [pbs[n]["_nrmse"][~pbs[n]["_is_res"]] for n in names]
    min_len = min(len(a) for a in arrs)
    mat = np.stack([a[:min_len] for a in arrs], axis=1)
    out = metrics.friedman_nemenyi(mat, names)
    ranked = sorted(out["avg_ranks"].items(), key=lambda kv: kv[1])
    print(f"cond {cond}: Friedman p={out['friedman_p']:.2e}  Nemenyi CD={out['nemenyi_critical_difference']:.3f}")
    print("  avg ranks (lower = better):", {k: round(v, 2) for k, v in ranked})
    print()


## Real-building evaluation

Two tracks:
1. **No-weather (headline)** — every real building, using the no-weather checkpoints. This is
   the primary real-world generalization result.
2. **+weather (temperature/humidity)** — the subset of real buildings whose dataset has published
   era5 weather on the public bucket (BuildingsBench's own real-eval only uses temperature; we
   include humidity too since it's already published alongside it). Buildings without published
   weather simply aren't in this track — no external fetching, no filled-in data.


In [ ]:
# === 8) Real-building evaluation: no-weather checkpoints (headline table, every real building) ===
def _r3(res, key):
    v = res.get(key, float("nan"))
    return round(v, 3) if v == v else ""

RFIELDS = ["model", "com_nrmse", "res_nrmse", "com_nmae", "res_nmae", "com_crps", "res_crps",
           "com_nrmse_pooled", "res_nrmse_pooled", "n_buildings", "sec"]
if not os.path.exists(paths.REAL_CSV):
    csv.DictWriter(open(paths.REAL_CSV, "w", newline=""), RFIELDS).writeheader()
rdone = set()
if os.path.getsize(paths.REAL_CSV) > 50:
    rdone = {r.model for _, r in pd.read_csv(paths.REAL_CSV).iterrows()}

for name in config.ALL_MODELS:
    if name in rdone:
        print(f"skip {name} (real-eval)")
        continue
    t0 = time.time()
    try:
        if name in ("xgboost", "lightgbm"):
            mdl, sigma = pickle.load(open(f"{paths.WEIGHTS_DIR}/{name}_A.pkl", "rb"))
            res = train.evaluate_real_tree(mdl, sigma, TE_REAL, load_transform, weather_cache=None, stride=24)
        else:
            base = models.build(name, L=config.L, H=config.H, n_time=TR["Ft"], n_weather=TR["Fw"],
                                 use_weather=False, **models.MODEL_KW.get(name, {})).to(DEV)
            if models.count_params(base) > 0:
                sd_path = f"{paths.WEIGHTS_DIR}/{name}_A.pt"
                if not os.path.exists(sd_path):
                    print(f"  {name}: no no-weather checkpoint yet, skip")
                    continue
                base.load_state_dict(torch.load(sd_path, map_location=DEV))
            print(f"real-eval {name}...", flush=True)
            res = train.evaluate_real(base, TE_REAL, DEV, load_transform, weather_cache=None,
                                       amp=config.USE_AMP and name not in config.RNN_MODELS, stride=24)
        os.makedirs(paths.PERBUILDING_REAL_DIR, exist_ok=True)
        pickle.dump({k: res[k] for k in ("_nrmse", "_nmae", "_crps", "_is_res") if k in res},
                    open(f"{paths.PERBUILDING_REAL_DIR}/{name}.pkl", "wb"))
        row = {"model": name,
               "com_nrmse": _r3(res, "Com NRMSE"), "res_nrmse": _r3(res, "Res NRMSE"),
               "com_nmae": _r3(res, "Com NMAE"), "res_nmae": _r3(res, "Res NMAE"),
               "com_crps": _r3(res, "Com CRPS"), "res_crps": _r3(res, "Res CRPS"),
               "com_nrmse_pooled": _r3(res, "Com NRMSE pooled"), "res_nrmse_pooled": _r3(res, "Res NRMSE pooled"),
               "n_buildings": res.get("n_buildings", 0), "sec": round(time.time() - t0)}
        csv.DictWriter(open(paths.REAL_CSV, "a", newline=""), RFIELDS).writerow(row)
        print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}  (n={row['n_buildings']})\n")
    except Exception as e:
        print(f"  FAILED {name}: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()
print("DONE ->", paths.REAL_CSV)


In [ ]:
# === 9) Real-building evaluation: +weather (temperature/humidity, wherever published) ===
RWFIELDS = ["model", "com_nrmse", "res_nrmse", "com_nmae", "res_nmae", "com_crps", "res_crps",
            "com_nrmse_pooled", "res_nrmse_pooled", "n_buildings", "n_weather_matched", "sec"]
if not os.path.exists(paths.REAL_WEATHER_TEMP_CSV):
    csv.DictWriter(open(paths.REAL_WEATHER_TEMP_CSV, "w", newline=""), RWFIELDS).writeheader()
rwdone = set()
if os.path.getsize(paths.REAL_WEATHER_TEMP_CSV) > 50:
    rwdone = {r.model for _, r in pd.read_csv(paths.REAL_WEATHER_TEMP_CSV).iterrows()}

wcache = weather_real.fetch_temp_humidity_track(paths, TE_REAL)
print(f"{len(wcache)}/{len(TE_REAL)} real buildings have published temperature/humidity")

for name in [m for m in config.ALL_MODELS if m not in ("xgboost", "lightgbm")]:
    if name in rwdone:
        print(f"skip {name} (+weather real-eval)")
        continue
    t0 = time.time()
    try:
        base = models.build(name, L=config.L, H=config.H, n_time=TR["Ft"], n_weather=TR["Fw"],
                             use_weather=True, **models.MODEL_KW.get(name, {})).to(DEV)
        if models.count_params(base) > 0:
            sd_path = f"{paths.WEIGHTS_DIR}/{name}_B.pt"
            if not os.path.exists(sd_path):
                print(f"  {name}: no +weather checkpoint yet, skip")
                continue
            base.load_state_dict(torch.load(sd_path, map_location=DEV))
        print(f"+weather real-eval {name}...", flush=True)
        res = train.evaluate_real(base, TE_REAL, DEV, load_transform, weather_cache=wcache,
                                   weather_transforms=weather_transforms,
                                   amp=config.USE_AMP and name not in config.RNN_MODELS, stride=24)
        os.makedirs(paths.PERBUILDING_REAL_WEATHER_TEMP_DIR, exist_ok=True)
        pickle.dump({k: res[k] for k in ("_nrmse", "_nmae", "_crps", "_is_res") if k in res},
                    open(f"{paths.PERBUILDING_REAL_WEATHER_TEMP_DIR}/{name}.pkl", "wb"))
        row = {"model": name,
               "com_nrmse": _r3(res, "Com NRMSE"), "res_nrmse": _r3(res, "Res NRMSE"),
               "com_nmae": _r3(res, "Com NMAE"), "res_nmae": _r3(res, "Res NMAE"),
               "com_crps": _r3(res, "Com CRPS"), "res_crps": _r3(res, "Res CRPS"),
               "com_nrmse_pooled": _r3(res, "Com NRMSE pooled"), "res_nrmse_pooled": _r3(res, "Res NRMSE pooled"),
               "n_buildings": res.get("n_buildings", 0), "n_weather_matched": res.get("n_weather_matched", 0),
               "sec": round(time.time() - t0)}
        csv.DictWriter(open(paths.REAL_WEATHER_TEMP_CSV, "a", newline=""), RWFIELDS).writerow(row)
        print(f"  done {row['sec']}s  Com NRMSE={row['com_nrmse']}  Res NRMSE={row['res_nrmse']}  (matched={row['n_weather_matched']})\n")
    except Exception as e:
        print(f"  FAILED {name}: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()
print("DONE ->", paths.REAL_WEATHER_TEMP_CSV)


## Final comparison: no-weather vs +weather on real buildings

Side-by-side Com NRMSE, matched by model, on the subset of real buildings both tracks actually
evaluated (only buildings with published weather appear in the +weather column — see cell 9's
match count above for how many that is per dataset).


In [ ]:
# === 10) Final side-by-side: does +weather (temp/humidity) help on real buildings? ===
real_nw = pd.read_csv(paths.REAL_CSV).set_index("model")
real_w = pd.read_csv(paths.REAL_WEATHER_TEMP_CSV).set_index("model")
print(f"{'model':24s} {'no-weather':>12s} {'+weather':>12s} {'delta':>8s} {'matched':>8s}")
for name in config.ALL_MODELS:
    if name in real_nw.index and name in real_w.index:
        a = real_nw.loc[name, "com_nrmse"]
        b = real_w.loc[name, "com_nrmse"]
        matched = real_w.loc[name, "n_weather_matched"]
        tag = "  (weather helps)" if b < a else ""
        print(f"{name:24s} {a:12.3f} {b:12.3f} {b - a:+8.3f} {matched:8.0f}{tag}")


## Weather-value analyses (the contribution layer)

Three analyses that turn "we re-benchmarked carefully" into a finding about *when and how much
weather inputs matter*. All run from saved checkpoints on Drive (no retraining), write resumable
CSVs to `results/analysis/`, and figures to `results/figures/`:

1. **Extreme days** — error bucketed by each building's own cold/mild/hot future-temperature
   deciles. Weather value concentrated on extreme days is invisible in annual averages.
2. **Longer horizons** — autoregressive rollout to 48h/72h. Copied-history information decays
   with horizon; known-future weather doesn't. Does the A-vs-B gap widen?
3. **Sensitivity gating** — per-channel knockout: which weather channels does each architecture
   actually use, and how much error does removing each one cost?


In [ ]:
# === 11) Weather-value 1: extreme-day evaluation (per-building cold/mild/hot temperature deciles) ===
# Averaging over a full year can hide weather value that's concentrated in exactly the days grid
# operators care about. Buckets every test window by the building's OWN future-temp deciles.
from bblab import analysis

EXT_CSV = f"{paths.ANALYSIS_DIR}/extreme_days.csv"
EFIELDS = ["model", "condition", "com_cold", "com_mild", "com_hot",
           "res_cold", "res_mild", "res_hot", "n_cold", "n_mild", "n_hot"]
if not os.path.exists(EXT_CSV):
    csv.DictWriter(open(EXT_CSV, "w", newline=""), EFIELDS).writeheader()
edone = {(r.model, r.condition) for _, r in pd.read_csv(EXT_CSV).iterrows()} if os.path.getsize(EXT_CSV) > 50 else set()

for name in config.ALL_MODELS:
    for cond, use_w in config.CONDITIONS.items():
        if (name, cond) in edone:
            print(f"skip {name}/{cond} (extreme-day)")
            continue
        fn = analysis.load_predict_fn(name, cond, use_w, TR["Ft"], TR["Fw"], paths, DEV)
        if fn is None:
            print(f"  {name}/{cond}: no checkpoint yet, skip")
            continue
        win = analysis.window_errors(fn, TE_SIM, DEV, use_w, load_transform, stride=24)
        ex = analysis.extreme_day_summary(win, TE_SIM)
        row = {"model": name, "condition": cond,
               "com_cold": round(ex.get("Com cold", float("nan")), 3),
               "com_mild": round(ex.get("Com mild", float("nan")), 3),
               "com_hot": round(ex.get("Com hot", float("nan")), 3),
               "res_cold": round(ex.get("Res cold", float("nan")), 3),
               "res_mild": round(ex.get("Res mild", float("nan")), 3),
               "res_hot": round(ex.get("Res hot", float("nan")), 3),
               "n_cold": ex["n cold"], "n_mild": ex["n mild"], "n_hot": ex["n hot"]}
        csv.DictWriter(open(EXT_CSV, "a", newline=""), EFIELDS).writerow(row)
        print(f"  {name}/{cond}: Com cold/mild/hot = {row['com_cold']}/{row['com_mild']}/{row['com_hot']}")

# headline: does weather help MORE on extreme days than mild ones?
edf = pd.read_csv(EXT_CSV)
if {"A", "B"} <= set(edf["condition"]):
    piv = edf.pivot_table(index="model", columns="condition",
                          values=["com_cold", "com_mild", "com_hot"])
    print("\nB-A delta (negative = weather helps) per bucket:")
    for bucket in ("com_cold", "com_mild", "com_hot"):
        d = piv[(bucket, "B")] - piv[(bucket, "A")]
        print(f"  {bucket:9s} mean Δ={d.mean():+.3f}  (models helped: {(d < 0).sum()}/{len(d)})")


In [ ]:
# === 12) Weather-value 2: autoregressive horizon rollout to 72h (A vs B) ===
# Feed each model's own 24h forecast back as history to reach 48h/72h. If known-future weather
# has value, the A-vs-B gap should WIDEN with horizon (persistence-copied information decays,
# weather doesn't). h1-24 reproduces the standard eval as a sanity anchor.
from bblab import analysis

ROLL_CSV = f"{paths.ANALYSIS_DIR}/rollout_horizons.csv"
segs = [f"h{24*k+1}-{24*(k+1)}" for k in range(config.ROLLOUT_STEPS)]
LFIELDS = ["model", "condition"] + [f"com_{s}" for s in segs] + [f"res_{s}" for s in segs]
if not os.path.exists(ROLL_CSV):
    csv.DictWriter(open(ROLL_CSV, "w", newline=""), LFIELDS).writeheader()
ldone = {(r.model, r.condition) for _, r in pd.read_csv(ROLL_CSV).iterrows()} if os.path.getsize(ROLL_CSV) > 50 else set()

for name in config.ALL_MODELS:
    for cond, use_w in config.CONDITIONS.items():
        if (name, cond) in ldone:
            print(f"skip {name}/{cond} (rollout)")
            continue
        fn = analysis.load_predict_fn(name, cond, use_w, TR["Ft"], TR["Fw"], paths, DEV)
        if fn is None:
            print(f"  {name}/{cond}: no checkpoint yet, skip")
            continue
        ro = analysis.rollout_eval(fn, TE_SIM, DEV, use_w, load_transform, stride=168)
        row = {"model": name, "condition": cond}
        for s in segs:
            row[f"com_{s}"] = round(ro[s]["Com"], 3)
            row[f"res_{s}"] = round(ro[s]["Res"], 3)
        csv.DictWriter(open(ROLL_CSV, "a", newline=""), LFIELDS).writerow(row)
        print(f"  {name}/{cond}: Com " + "  ".join(f"{s}={row[f'com_{s}']}" for s in segs))

# does the weather advantage grow with horizon?
rdf = pd.read_csv(ROLL_CSV)
if {"A", "B"} <= set(rdf["condition"]):
    piv = rdf.pivot_table(index="model", columns="condition", values=[f"com_{s}" for s in segs])
    print("\nB-A delta by horizon (negative = weather helps; watch whether it grows):")
    for s in segs:
        d = piv[(f"com_{s}", "B")] - piv[(f"com_{s}", "A")]
        print(f"  {s:8s} mean Δ={d.mean():+.3f}  (models improving with weather: {(d < 0).sum()}/{len(d)})")


In [ ]:
# === 13) Weather-value 3: per-channel sensitivity gating (condition-B checkpoints) ===
# Knock out one weather channel at a time (-> training mean) over the forecast horizon and
# measure prediction movement + NRMSE degradation. Models x channels heatmap saved to Drive.
from bblab import analysis
import matplotlib.pyplot as plt

SENS_CSV = f"{paths.ANALYSIS_DIR}/weather_sensitivity.csv"
SFIELDS = ["model", "base_nrmse"] + [f"{c}_dnrmse" for c in config.WEATHER_COLS] + [f"{c}_dpred" for c in config.WEATHER_COLS]
if not os.path.exists(SENS_CSV):
    csv.DictWriter(open(SENS_CSV, "w", newline=""), SFIELDS).writeheader()
sdone = set(pd.read_csv(SENS_CSV)["model"]) if os.path.getsize(SENS_CSV) > 50 else set()

for name in [m for m in config.ALL_MODELS if m not in config.PERSISTENCE_MODELS]:
    if name in sdone:
        print(f"skip {name} (sensitivity)")
        continue
    fn = analysis.load_predict_fn(name, "B", True, TR["Ft"], TR["Fw"], paths, DEV)
    if fn is None:
        print(f"  {name}/B: no checkpoint yet, skip")
        continue
    sens = analysis.weather_sensitivity(fn, TE_SIM, DEV, load_transform, n_windows=4096)
    row = {"model": name, "base_nrmse": round(sens["_base_nrmse"], 3)}
    for c in config.WEATHER_COLS:
        row[f"{c}_dnrmse"] = round(sens[c]["delta_nrmse"], 4)
        row[f"{c}_dpred"] = round(sens[c]["pred_delta_pct"], 3)
    csv.DictWriter(open(SENS_CSV, "a", newline=""), SFIELDS).writerow(row)
    print(f"  {name}: base={row['base_nrmse']}  temp ΔNRMSE={row['temperature_dnrmse']:+.3f}")

# heatmap: models x channels, NRMSE degradation when the channel is removed
sdf = pd.read_csv(SENS_CSV).set_index("model")
if len(sdf):
    mat = sdf[[f"{c}_dnrmse" for c in config.WEATHER_COLS]].values
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(sdf) + 1.5), constrained_layout=True)
    vmax = max(1e-6, np.nanpercentile(np.abs(mat), 98))
    im = ax.imshow(mat, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(config.N_WEATHER), [c.replace("_", "\n") for c in config.WEATHER_COLS], fontsize=8)
    ax.set_yticks(range(len(sdf)), sdf.index, fontsize=8)
    fig.colorbar(im, ax=ax, label="ΔNRMSE when channel removed (+ = channel was helping)")
    ax.set_title("Weather-channel sensitivity (condition B, simulated test)")
    fig.savefig(f"{paths.FIGURES_DIR}/sensitivity_heatmap.png", dpi=200)
    plt.show()


In [ ]:
# === 14) Compute budget: params, epochs, wall time, peak GPU memory (all persisted in sim_results_v3.csv on Drive) ===
budget = pd.read_csv(paths.SIM_CSV)
cols = ["model", "condition", "params", "epochs", "train_sec", "eval_sec", "peak_mem_gb", "gpu", "reused_ckpt"]
display(budget[cols].sort_values(["model", "condition"]).reset_index(drop=True))
total_h = budget["train_sec"].sum() / 3600
print(f"total training compute this file: {total_h:.2f} GPU-hours on {budget['gpu'].iloc[0] if len(budget) else '?'}")
print("NOTE: rows with reused_ckpt=True were trained in an earlier session; their train_sec/epochs")
print("are recorded in the results file of the session that trained them (v1/v2 files on Drive).")
